# tp-19b 追い切りラップの特徴量（拡張）: 調教師内の相対化と絞り込み

tp-19 で TL_A（標準化した追い切りの時計・加速度・日数・脚色）が勝率 −0.00136 で採用になった。ここでは次の拡張を、TL_A に対して比べる（`trainer_id` をキー、前日までに確定したレースだけ）。

| 群 | 内容 |
|----|------|
| TL_A | 基準（tp-19 の採用セット） |
| TL_AT | TL_A + 調教師内の相対化: その調教師の過去の出走の平均に対する、最終追い切りの時計（4F・終い1F）・加速度・日数・強度・1 週前の有無・本数の差と、過去の件数（厩舎の「型外」度） |
| TL_core | TL_A のうち寄与の大きい 12 列だけ（少ない列で同じ効果か） |

評価は、基準（TL なし）に対する差と、TL_A に対する差（TL_AT・TL_core が TL_A を超えるか）の両方。LightGBM、valid 2022〜2024 × 2 seed、test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, zoo, zoo_runner, feature_bakeoff as fb
from src.pipeline.models.compute import detect_compute
from src.pipeline.features import training_patterns as tp

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OUT = WORK / "training_laps_b"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP19B_THREADS", detect_compute(reserve_cores=6).threads))
SEEDS = [0, 1]
SMOKE = os.environ.get("TP19B_SMOKE") == "1"
mlflow.set_experiment("keiba-tp-training-laps")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X0 = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP = zoo.tuned_params("lgb", best["lgb"]["params"])
if SMOKE:
    folds = folds[:1]
key = ["race_id", "horse_id"]
if "trainer_id" not in df.columns:
    tt = pd.read_parquet(WORK / "features/train_table.parquet", columns=key + ["trainer_id"]).astype({"race_id": str, "horse_id": str})
    df = df.merge(tt, on=key, how="left")
RD = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
print("行", len(df), "| 調教師 ID の欠損", int(df["trainer_id"].isna().sum()), "| smoke", SMOKE)

## 追い切り表の読み込みと、時計の標準化

In [ ]:
laps = pd.read_parquet(WORK / "features/training_laps.parquet")
print("追い切り", len(laps), "本 | 馬", laps["horse_id"].nunique(), "| 種別", laps["kind"].value_counts().to_dict())
t = tp.add_reference_z(tp.prepare(laps))
print("標準化できた割合（時計あり）:", round(float(t.loc[t['timed'], 't4z'].notna().mean()), 3))
feat = tp.add_self_relative(tp.race_features(t))
M = pd.DataFrame({"horse_id": df["horse_id"].astype(str).to_numpy(), "race_date": RD.to_numpy()}, index=df.index).merge(feat, on=["horse_id", "race_date"], how="left")
M.index = df.index
cov = M["has_training"].notna().groupby(pd.to_datetime(df["date"]).dt.year).mean()
print("レースへの対応率（年別）:", cov.round(3).to_dict())
print("最終追い切り（時計あり）の割合:", round(float(M["f_t1"].notna().mean()), 3), "| 1週前あり:", round(float(M["w1_exists"].fillna(0).mean()), 3))

## 特徴量群を作る

In [ ]:
A_COLS = [c for c in M.columns if c.startswith(("f_", "w1_", "w2_")) and c not in ("f_untimed_later",)] + ["f_untimed_later", "n_work_14", "n_hard_14", "mean_t4z_14", "best_t1z_14", "n_work_28", "slope_share_28", "best_t1z_28", "has_training"]
A_COLS = [c for c in dict.fromkeys(A_COLS) if c in M.columns and c not in ("horse_id", "race_date")]
TR = tp.trainer_relative_features(M, df["trainer_id"], df["date"])
CORE = ["best_t1z_28", "best_t1z_14", "mean_t4z_14", "f_t1z", "f_t1z_vs_self", "w1_t4", "f_pos", "f_t4z", "f_t4", "w2_t1z", "f_rank", "f_t4z_vs_self"]
G = {"TL_A": M[A_COLS].astype("float32")}
G["TL_AT"] = pd.concat([G["TL_A"], TR.astype("float32")], axis=1)
G["TL_core"] = M[[c for c in CORE if c in M.columns]].astype("float32")
print("trainer_id 欠損", int(df["trainer_id"].isna().sum()), "| 調教師内の相対化の欠損率", round(float(TR.isna().mean().mean()), 3))
for k_, v_ in G.items():
    v_.assign(race_id=df["race_id"].to_numpy(), horse_id=df["horse_id"].to_numpy()).to_parquet(OUT / f"cand_{k_}.parquet", index=False)
print({k_: v_.shape[1] for k_, v_ in G.items()})

## 基準との比較（LightGBM、2 seed）

In [ ]:
fin_map = meta.set_index(key)["finish_position"]
res0, OOF0 = zoo_runner.run_oof("lgb_tl_base", "lgb_binary", X0, meta, folds, SEEDS, THREADS, params=dict(LP))
print(f"基準: win {res0['logloss_win'].mean():.5f} top2 {res0['logloss_top2'].mean():.5f} top3 {res0['logloss_top3'].mean():.5f}", flush=True)
ROWS, ROWS_VSA, OOFS = [], [], {}
names = list(G) if not SMOKE else ["TL_A", "TL_core"]
for name in names:
    t0 = time.time()
    X = pd.concat([X0, G[name].add_prefix(f"tl_{name}_")], axis=1)
    res, oof = zoo_runner.run_oof(f"lgb_tl_{name}", "lgb_binary", X, meta, folds, SEEDS, THREADS, params=dict(LP))
    OOFS[name] = oof
    ROWS.append(fb.compare_oof(OOF0, oof, fin_map, name))
    r = ROWS[-1]
    if name != "TL_A" and "TL_A" in OOFS:
        ra = fb.compare_oof(OOFS["TL_A"], oof, fin_map, name + " vs TL_A")
        ROWS_VSA.append(ra)
        print(f"   {name} vs TL_A: Δwin {ra['Δwin']:+.5f} (上限 {ra['win_hi']:+.5f}) top2 {ra['Δtop2']:+.5f} top3 {ra['Δtop3']:+.5f}", flush=True)
    print(f"{name}: win {res['logloss_win'].mean():.5f} Δ {r['Δwin']:+.5f} (上限 {r['win_hi']:+.5f}) top2 {r['Δtop2']:+.5f} top3 {r['Δtop3']:+.5f} 採用={r['採用']} ({time.time() - t0:.0f}s)", flush=True)
    pd.DataFrame(ROWS).set_index("variant").to_csv(OUT / "tl_vs_base.csv")
    gc.collect()
R = pd.DataFrame(ROWS).set_index("variant")
display(R[["win_base", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5))
if ROWS_VSA:
    RA = pd.DataFrame(ROWS_VSA).set_index("variant"); display(RA[["win_base", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5)); RA.to_csv(OUT / "tl_vs_A.csv")

In [ ]:
import lightgbm as lgb
y = (meta["finish_position"].to_numpy() == 1).astype(int)
bn = R.sort_values("Δwin").index[0]
Xb = pd.concat([X0, G[bn].add_prefix(f"tl_{bn}_")], axis=1)
f = folds[-1]
cats = [c for c in Xb.columns if str(Xb[c].dtype) == "category"]
prm = dict({"objective": "binary", "verbosity": -1, "num_threads": THREADS, "seed": 0, "bagging_freq": 1}, **LP["lgb"])
bst = lgb.train(prm, lgb.Dataset(Xb.iloc[f.train_index], label=y[f.train_index], categorical_feature=cats), num_boost_round=LP["n_rounds"])
g = pd.Series(bst.feature_importance("gain"), index=Xb.columns); g = g / g.sum()
new = [c for c in Xb.columns if c.startswith("tl_")]
print("最良:", bn, "| 新特徴量の gain 合計", round(float(g[new].sum()) * 100, 2), "%"); display(g[new].sort_values(ascending=False).head(12).round(4))
adopted = list(R.index[R["採用"]])
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "coverage_by_year": cov.round(4).to_dict(), "results": R.round(6).reset_index().to_dict("records"), "vs_A": (RA.round(6).reset_index().to_dict("records") if ROWS_VSA else []),
                                             "adopted": adopted, "best": bn, "gain_share_pct": float(g[new].sum()) * 100}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name="tp-19b-training-laps", tags={"notebook": "tp-19b", "git_commit": GIT_COMMIT}):
    for i_, n in enumerate(R.index):
        mlflow.log_metrics({f"delta_win_g{i_}": float(R.loc[n, "Δwin"])})
    mlflow.log_dict({f"g{i_}": n for i_, n in enumerate(R.index)}, "training_laps/metric_index.json")
    for f_ in [x for x in ("tl_vs_base.csv", "tl_vs_A.csv", "report.json") if (OUT / x).exists()]:
        mlflow.log_artifact(str(OUT / f_), "training_laps")